# Part 1: Conceptual Check (15 min)
Test your understanding before diving into the code.

1. What is the key difference between a Pandas Series and a DataFrame?

2. You run df.loc[0:3]. How many rows do you get? What about df.iloc[0:3]?

3. You have a DataFrame with a price column stored as object type (strings like "£45.99"). What problem does this cause, and how do you fix it?

4. When would you use .apply() instead of a direct vectorised operation like df['col'] * 2?

5. How do you select all numeric columns from a DataFrame, excluding float types?

# Answer
1. A Pandas Series is a 1D array similar to python array. Pandas DataFrame is similar to a SQL table, with indexes and columns
2. Running df.loc[0:3] returns 4 rows from index 0 - 4 (inclusive). Running df.iloc[0:3] returns 3 rows from index 0 - 2 (exclusive)
3. If price is stored as an object type you cant perform math on it. You can str.replace the pound symbol and then chain convert it to a float using df['price'].astype(float)
    * df['price] = df['price].str.replace('£', '').astype(float)
4. When you want to apply a custom function or when a vectorised operation is not available
5. int_cols = df.select_dtypes(include='int')

# Part 2: Practical Challenge (30–45 min)
You're a data analyst for an online bookshop. The operations team has exported their inventory to a CSV and they need your help making sense of it.

In [259]:
# Challenge 1
# Create the following DataFrame and perform a health check

import pandas as pd

books = pd.DataFrame({
    'title': ['The Pragmatic Programmer', 'Clean Code', 'Python Crash Course',
              'Data Science from Scratch', 'Designing Data-Intensive Apps'],
    'author': ['Hunt & Thomas', 'Robert Martin', 'Eric Matthes',
               'Joel Grus', 'Martin Kleppmann'],
    'genre': ['Tech', 'Tech', 'Tech', 'Data', 'Data'],
    'price': [45.99, 38.50, 29.99, 42.00, 55.00],
    'units_sold': [120, 95, 200, 80, 60],
    'in_stock': [True, True, False, True, False]
})

# 1. Use .info() — how many non-null values does each column have?

books.info()

    # A1) Each value has 5 non-null values


# 2. Use .describe() — what's the average price and maximum units sold?

books.describe()

    # A2) Average price:  42.296
    #     Max units sold: 200 
    


# 3. Add a revenue column: price × units_sold.

books['revenue'] = books['price'] * books['units_sold']


# 4. What is the total revenue across all books?

print(books['revenue'].sum())

    # A4) Total Revenue across all books: 21834.3

books

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   title       5 non-null      object 
 1   author      5 non-null      object 
 2   genre       5 non-null      object 
 3   price       5 non-null      float64
 4   units_sold  5 non-null      int64  
 5   in_stock    5 non-null      bool   
dtypes: bool(1), float64(1), int64(1), object(3)
memory usage: 337.0+ bytes
21834.3


,title,author,genre,price,units_sold,in_stock,revenue
0,The Pragmatic Programmer,Hunt & Thomas,Tech,45.99,120,True,5518.8
1,Clean Code,Robert Martin,Tech,38.50,95,True,3657.5
2,Python Crash Course,Eric Matthes,Tech,29.99,200,False,5998.0
3,Data Science from Scratch,Joel Grus,Data,42.00,80,True,3360.0
4,Designing Data-Intensive Apps,Martin Kleppmann,Data,55.00,60,False,3300.0


In [260]:
# Challenge 2

# 1. Use .loc to select only title and revenue for Data genre books.
# mask = books['genre'] == "Data", ['title', 'revenue']
display(books.loc[books['genre'] == "Data", ['title', 'revenue']]) # inline method

# 2. Use .iloc to retrieve the last 3 rows and first 3 columns.
display(books.iloc[-3:, 0:3])

# 3. Filter books that are in stock AND have a price above £40.
mask = ((books['in_stock'] == True) & (books['price'] > 40))
display(books[mask])

# 4. Find the book with the highest revenue using .loc with .idxmax().
books.loc[books['revenue'].idxmax()]

,title,revenue
3,Data Science from Scratch,3360.0
4,Designing Data-Intensive Apps,3300.0


,title,author,genre
2,Python Crash Course,Eric Matthes,Tech
3,Data Science from Scratch,Joel Grus,Data
4,Designing Data-Intensive Apps,Martin Kleppmann,Data


,title,author,genre,price,units_sold,in_stock,revenue
0,The Pragmatic Programmer,Hunt & Thomas,Tech,45.99,120,True,5518.8
3,Data Science from Scratch,Joel Grus,Data,42.00,80,True,3360.0


title         Python Crash Course
author               Eric Matthes
genre                        Tech
price                       29.99
units_sold                    200
in_stock                    False
revenue                    5998.0
Name: 2, dtype: object

# Challenge 3: Transform, Sort & Rank

In [261]:
# 1. Use .apply() to add a price_tier column: 'Budget' (< £35), 'Standard' (£35–£50), 'Premium' (> £50).
def tier(price) -> str:
    if price > 50:
        return 'Premium'
    elif ((price >= 35) & (price <= 50)):
        return 'Standard'
    else:
        return 'Budget'
        

books['price_tier'] = books['price'].apply(tier)
display(books)

# 2. Sort the DataFrame by revenue descending.

books.sort_values(by='revenue', ascending=False)
display(books)

# 3. Add a sales_rank column using .rank(ascending=False).

books['sales_rank'] = books['revenue'].rank(method='min', ascending=False).astype(int)
display(books)


# 4. Use .map() to encode genre as a number: {'Tech': 0, 'Data': 1}.
encode = {'Tech': 0,
          'Data': 1}
books['genre'] = books['genre'].map(encode)
display(books)



,title,author,genre,price,units_sold,in_stock,revenue,price_tier
0,The Pragmatic Programmer,Hunt & Thomas,Tech,45.99,120,True,5518.8,Standard
1,Clean Code,Robert Martin,Tech,38.50,95,True,3657.5,Standard
2,Python Crash Course,Eric Matthes,Tech,29.99,200,False,5998.0,Budget
3,Data Science from Scratch,Joel Grus,Data,42.00,80,True,3360.0,Standard
4,Designing Data-Intensive Apps,Martin Kleppmann,Data,55.00,60,False,3300.0,Premium


,title,author,genre,price,units_sold,in_stock,revenue,price_tier
0,The Pragmatic Programmer,Hunt & Thomas,Tech,45.99,120,True,5518.8,Standard
1,Clean Code,Robert Martin,Tech,38.50,95,True,3657.5,Standard
2,Python Crash Course,Eric Matthes,Tech,29.99,200,False,5998.0,Budget
3,Data Science from Scratch,Joel Grus,Data,42.00,80,True,3360.0,Standard
4,Designing Data-Intensive Apps,Martin Kleppmann,Data,55.00,60,False,3300.0,Premium


,title,author,genre,price,units_sold,in_stock,revenue,price_tier,sales_rank
0,The Pragmatic Programmer,Hunt & Thomas,Tech,45.99,120,True,5518.8,Standard,2
1,Clean Code,Robert Martin,Tech,38.50,95,True,3657.5,Standard,3
2,Python Crash Course,Eric Matthes,Tech,29.99,200,False,5998.0,Budget,1
3,Data Science from Scratch,Joel Grus,Data,42.00,80,True,3360.0,Standard,4
4,Designing Data-Intensive Apps,Martin Kleppmann,Data,55.00,60,False,3300.0,Premium,5


,title,author,genre,price,units_sold,in_stock,revenue,price_tier,sales_rank
0,The Pragmatic Programmer,Hunt & Thomas,0,45.99,120,True,5518.8,Standard,2
1,Clean Code,Robert Martin,0,38.50,95,True,3657.5,Standard,3
2,Python Crash Course,Eric Matthes,0,29.99,200,False,5998.0,Budget,1
3,Data Science from Scratch,Joel Grus,1,42.00,80,True,3360.0,Standard,4
4,Designing Data-Intensive Apps,Martin Kleppmann,1,55.00,60,False,3300.0,Premium,5


In [262]:
# Stretch Challenge
# The manager wants a report comparing average price and total revenue by price_tier. 
# Use groupby() to generate this summary. 
# Which tier contributes the most total revenue despite having the lowest average price?

summary = books.groupby('price_tier').agg(
    average_price=('price', 'mean'),
    total_revenue=('revenue', 'sum'),
    total_units_sold=('units_sold', 'sum')
)


display(summary.sort_values('average_price'))

# Standard Tier contributes the most total revenue, but it depends on the cost of acquiring the books as well to accurately calculate the profits
# Budget Tier has the lowest average price and more total total units sold than Premium Tier giving the 2nd highest revenue

,average_price,total_revenue,total_units_sold
price_tier,,,
Budget,29.990000,5998.0,200
Standard,42.163333,12536.3,295
Premium,55.000000,3300.0,60


Practicing groupby havent used it in awhile

In [263]:
books.groupby('price_tier')['revenue'].sum()

price_tier
Budget       5998.0
Premium      3300.0
Standard    12536.3
Name: revenue, dtype: float64

In [264]:
books.groupby('price_tier')['price'].mean()

price_tier
Budget      29.990000
Premium     55.000000
Standard    42.163333
Name: price, dtype: float64

In [265]:
books.groupby('price_tier')['units_sold'].max()

price_tier
Budget      200
Premium      60
Standard    120
Name: units_sold, dtype: int64

In [266]:
books.groupby('price_tier')['title'].count()

price_tier
Budget      1
Premium     1
Standard    3
Name: title, dtype: int64